# RegionCLIP trên box YOLO cố định
Chọn 1 GPU T4 và bật Internet. Tải **regionclip_pretrained-cc_rn50.pth** từ [Model Zoo chính thức](https://github.com/microsoft/RegionCLIP/blob/4b8513b56e24827e3d6468e1f2105869f35c2d0b/docs/MODEL_ZOO.md), gắn Dataset chứa file qua Add Input. Notebook tự tìm file; điền CHECKPOINT_PATH nếu có nhiều bản.
Lượt này trích bốn vector vùng trên hai ảnh VALID. Gửi **regionclip_fixed_roi_valid2_review.zip**.
Có thể đổi đuôi checkpoint thành **.bin** trước khi upload. Notebook nhận cả hai tên.


In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile, zipfile
CHECKPOINT_PATH = ''  # Optional full path to the mounted official RN50 checkpoint.
FILENAMES = ('regionclip_pretrained-cc_rn50.pth', 'regionclip_pretrained-cc_rn50.bin')
REVISION = '4b8513b56e24827e3d6468e1f2105869f35c2d0b'
if CHECKPOINT_PATH:
    CHECKPOINT = Path(CHECKPOINT_PATH)
    if not CHECKPOINT.is_file() or CHECKPOINT.name not in FILENAMES:
        raise FileNotFoundError(f'Expected an RN50 checkpoint at {CHECKPOINT}')
else:
    candidates = sorted(path for name in FILENAMES for path in Path('/kaggle/input').rglob(name) if path.is_file())
    if not candidates:
        raise FileNotFoundError(f'Attach a Kaggle Dataset containing {FILENAMES[0]} or {FILENAMES[1]} via Add Input, then rerun this cell.')
    if len(candidates) > 1:
        raise ValueError('Set CHECKPOINT_PATH to one of these matches: '+', '.join(map(str,candidates)))
    CHECKPOINT = candidates[0]
print('RegionCLIP checkpoint:', CHECKPOINT, flush=True)
OUTPUT = Path('/kaggle/working/regionclip_fixed_roi_valid2')
OUTPUT.mkdir(exist_ok=False)
ROOT=Path(tempfile.mkdtemp(prefix='regionclip-fixed-roi-'))
REPO=ROOT/'RegionCLIP'
DEPS=ROOT/'deps'
env=dict(os.environ,MAX_JOBS='2',CUDA_VISIBLE_DEVICES='')
def run(args, timeout, environment=None, cwd=None):
    with (OUTPUT/'notebook.log').open('a',encoding='utf8') as log:
        p=subprocess.Popen([str(x) for x in args],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                           text=True,encoding='utf8',errors='replace',env=environment,cwd=cwd)
        try:
            result, _=p.communicate(timeout=timeout)
        except subprocess.TimeoutExpired:
            p.kill()
            result, _=p.communicate()
            log.write(result)
            raise
        log.write(result)
        print(result[-6000:])
        if p.returncode:
            raise subprocess.CalledProcessError(p.returncode,args)


In [ ]:
WORKER_SOURCE="CASES=[{'coco_id': 239894, 'image_sha256': '79cfff40fc14c76c7bbe9a409b91ef1b03ec7759cb8e6ea94b277e49f810828b', 'rois': [{'roi': 1, 'label': 'teddy bear', 'confidence': 0.926, 'bbox': [0.5461971830985916, 0.43859375, 0.793169014084507, 0.704875]}, {'roi': 6, 'label': 'handbag', 'confidence': 0.5338, 'bbox': [0.16483568075117372, 0.733359375, 0.4577230046948357, 0.9990937499999999]}]}, {'coco_id': 487438, 'image_sha256': '1921b1a7256c483720117c5aca5bc1ce67dfd73fff8132fbec639167edfd77b7', 'rois': [{'roi': 1, 'label': 'bed', 'confidence': 0.8526, 'bbox': [0.160296875, 0.48352083333333334, 0.7308125000000001, 0.9886666666666667]}, {'roi': 7, 'label': 'chair', 'confidence': 0.3943, 'bbox': [0.910171875, 0.6130208333333333, 0.999578125, 0.9525208333333333]}]}]\nREVISION='4b8513b56e24827e3d6468e1f2105869f35c2d0b'\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\nimport time\nimport urllib.request\n\nimport numpy as np\nimport torch\nfrom detectron2.config import get_cfg\nfrom detectron2.data import detection_utils as utils\nfrom detectron2.data import transforms as T\nfrom detectron2.modeling import build_backbone\nfrom detectron2.structures import Boxes, ImageList\nfrom detectron2.modeling.poolers import ROIPooler\n\nrepo, checkpoint, out = map(Path,sys.argv[1:])\nif not torch.cuda.is_available():\n    raise RuntimeError('Select a CUDA GPU')\ndevice=torch.device('cuda:0')\ncfg=get_cfg()\ncfg.merge_from_file(str(repo/'configs/LVISv1-InstanceSegmentation/CLIP_fast_rcnn_R_50_C4_zsinf.yaml'))\nbackbone=build_backbone(cfg)\nif not hasattr(backbone,'attnpool'):\n    raise ValueError('Expected RegionCLIP RN50 attention pooling')\nwith checkpoint.open('rb') as source:\n    checkpoint_sha=hashlib.file_digest(source,'sha256').hexdigest()\npayload=torch.load(checkpoint,map_location='cpu',weights_only=False)\nstate=payload['model']\nweights={}\nfor key,value in state.items():\n    key=key.removeprefix('module.')\n    if key.startswith('backbone.'):\n        weights[key.removeprefix('backbone.')]=torch.as_tensor(value)\nbackbone.load_state_dict(weights,strict=True)\ndel payload,state,weights\nbackbone.to(device).eval()\npooler=ROIPooler(output_size=cfg.MODEL.ROI_BOX_HEAD.POOLER_RESOLUTION,\n                scales=(1/backbone.output_shape()['res4'].stride,),\n                sampling_ratio=cfg.MODEL.ROI_BOX_HEAD.POOLER_SAMPLING_RATIO,\n                pooler_type=cfg.MODEL.ROI_BOX_HEAD.POOLER_TYPE).to(device)\nmean=torch.tensor(cfg.MODEL.PIXEL_MEAN,device=device)[:,None,None]\nstd=torch.tensor(cfg.MODEL.PIXEL_STD,device=device)[:,None,None]\nif max(cfg.MODEL.PIXEL_MEAN)>=1:\n    raise ValueError('Unexpected image normalization')\narrays={}\nrecords=[]\nstarted=time.monotonic()\nwith torch.inference_mode():\n    for case in CASES:\n        cid=case['coco_id']\n        filename=f'COCO_val2014_{cid:012d}.jpg'\n        url='https://s3.amazonaws.com/images.cocodataset.org/val2014/'+filename\n        with urllib.request.urlopen(url,timeout=30) as response:\n            data=response.read()\n        if hashlib.sha256(data).hexdigest()!=case['image_sha256']:\n            raise ValueError(f'Image SHA differs for {cid}')\n        path=out/filename\n        path.write_bytes(data)\n        image=utils.read_image(str(path),format=cfg.INPUT.FORMAT)\n        augmented=T.AugInput(image)\n        transform=T.AugmentationList(utils.build_augmentation(cfg,False))(augmented)\n        h,w=image.shape[:2]\n        raw=np.asarray([row['bbox'] for row in case['rois']],dtype=np.float64)*[w,h,w,h]\n        if not np.isfinite(raw).all() or not (raw[:,2:]>raw[:,:2]).all():\n            raise ValueError('Invalid YOLO boxes')\n        shifted=transform.apply_box(raw)\n        x=torch.as_tensor(np.ascontiguousarray(augmented.image.transpose(2,0,1)),device=device).float()\n        batch=ImageList.from_tensors([(x/255-mean)/std],backbone.size_divisibility)\n        spatial=backbone(batch.tensor)['res4']\n        boxes=Boxes(torch.as_tensor(shifted,dtype=torch.float32,device=device))\n        vector=backbone.attnpool(backbone.layer4(pooler([spatial],[boxes])))\n        if vector.ndim!=2 or vector.shape!=(2,1024) or not torch.isfinite(vector).all():\n            raise ValueError('Unexpected RegionCLIP RN50 vectors')\n        if torch.equal(vector[0],vector[1]) or (vector.norm(dim=1)<=0).any():\n            raise ValueError('Degenerate vectors')\n        arrays[str(cid)]=vector.float().cpu().numpy()\n        records.append(dict(coco_id=cid,rois=case['rois'],image_sha256=case['image_sha256'],\n                            feature_shape=list(vector.shape),resized_xyxy=shifted.tolist()))\nnp.savez(out/'region_features.npz',**arrays)\n(out/'result.json').write_text(json.dumps(dict(checkpoint_sha256=checkpoint_sha,source_revision=REVISION,\n                                              seconds=time.monotonic()-started,records=records),indent=2))\n"
WORKER = ROOT/'run.py'
WORKER.write_text(WORKER_SOURCE,encoding='utf8')
try:
    run(['git','init','-q',REPO],30)
    run(['git','-C',REPO,'fetch','--depth','1','https://github.com/microsoft/RegionCLIP.git',REVISION],180)
    run(['git','-C',REPO,'checkout','--detach',REVISION],60)
    run([sys.executable,'-m','pip','install','--target',DEPS,
         'yacs','fvcore','iopath','termcolor','diffdist','ftfy','pycocotools','omegaconf'],240)
    env['PYTHONPATH']=str(DEPS)
    run([sys.executable,'setup.py','build_ext','--inplace'],480,env,REPO)
    inference_env=dict(os.environ,PYTHONPATH=os.pathsep.join((str(REPO),str(DEPS))))
    run([sys.executable,WORKER,REPO,CHECKPOINT,OUTPUT],240,inference_env)
finally:
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for name in ('notebook.log','result.json','region_features.npz'):
            if (OUTPUT/name).exists():
                z.write(OUTPUT/name,name)
    print('SEND REVIEW',review)
